In [ ]:
# Update of AmbrosMs' great notebook
# Removes the constant FPS assumption; handles variable frame timing
# Added full GPU suppoprt and extra trees/feats

verbose = True

import pandas as pd
import numpy as np
from tqdm import tqdm
import itertools
import warnings
import json
import hashlib
import os, random
import gc, re, math
import lightgbm
from collections import defaultdict
import polars as pl
from scipy import signal, stats
from typing import Dict, Optional, Tuple
from time import perf_counter 
from sklearn.base import ClassifierMixin, BaseEstimator, clone
from sklearn.model_selection import cross_val_predict, GroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')
USE_GPU = ("KAGGLE_KERNEL_RUN_TYPE" in __import__("os").environ) and (__import__("shutil").which("nvidia-smi") is not None)
print(f'Using GPU? {USE_GPU}')

from xgboost import XGBClassifier, XGBRFClassifier
from catboost import CatBoostClassifier
 
SEED = 1234

os.environ["MY_FOLD"]="0"

In [ ]:
# --- SEED EVERYTHING -----
os.environ["PYTHONHASHSEED"] = str(SEED)      # has to be set very early

rnd = np.random.RandomState(SEED)
random.seed(SEED)
np.random.seed(SEED)

def _make_lgbm(**kw):
    kw.setdefault("random_state", SEED)
    # kw.setdefault("deterministic", True)
    # kw.setdefault("force_row_wise", True) 
    kw.setdefault("feature_fraction_seed", SEED)
    kw.setdefault("data_random_seed", SEED)
    kw.setdefault("device", 'gpu' if USE_GPU else 'cpu')
    return lightgbm.LGBMClassifier(**kw)

def _make_xgb(**kw):
    kw.setdefault("random_state", SEED)
    kw.setdefault("tree_method", "gpu_hist" if USE_GPU else "hist")
    # kw.setdefault("deterministic_histogram", True)
    return XGBClassifier(**kw)

def _make_cb(**kw):
    kw.setdefault("random_seed", SEED)
    if USE_GPU:
        kw.setdefault("task_type", "GPU")
        kw.setdefault("devices", "0")
    else:
        kw.setdefault("task_type", "CPU")

    return CatBoostClassifier(**kw)


In [ ]:
# ================= StratifiedSubsetClassifier =================
class StratifiedSubsetClassifierWEval(ClassifierMixin, BaseEstimator):
    def __init__(self,
                 estimator,
                 n_samples=None,
                 random_state: int = 42,
                 valid_size: float = 0.10,
                 val_cap_ratio: float = 0.25,
                 es_rounds: "int|str" = "auto",
                 es_metric: str = "auto"):
        self.estimator = estimator
        self.n_samples = (int(n_samples) if (n_samples is not None) else None)
        self.random_state = random_state
        self.valid_size = float(valid_size)
        self.val_cap_ratio = float(val_cap_ratio)
        self.es_rounds = es_rounds
        self.es_metric = es_metric
 
    # -------------------------- API --------------------------
    def fit(self, X: pd.DataFrame, y):
        y = np.asarray(y)
        n_total = len(y); assert n_total == len(X)

        tr_idx, va_idx = self._compute_train_val_indices(y, n_total)
        Xtr = X.iloc[tr_idx]; ytr = y[tr_idx]

        Xtr = Xtr.to_numpy(np.float32, copy=False)

        Xva = yva = None
        if va_idx is not None and len(va_idx) > 0:
            Xva = X.iloc[va_idx].to_numpy(np.float32, copy=False); yva = y[va_idx]

        # Compute pos_rate on VALIDATION (what ES monitors)
        pos_rate = None
        if yva is not None and len(yva) > 0:
            pos_rate = float(np.mean(yva == 1))

        # Decide metric & patience
        metric = self._choose_metric(pos_rate)
        patience = self._choose_patience(pos_rate)

        # Apply imbalance knobs per library
        if self._is_xgb(self.estimator):
            # scale_pos_weight = n_neg / n_pos on TRAIN
            n_pos = max(1, int((ytr == 1).sum()))
            n_neg = max(1, len(ytr) - n_pos)
            self.estimator.set_params(scale_pos_weight=(n_neg / n_pos))
            self.estimator.set_params(eval_metric=metric)

        elif self._is_catboost(self.estimator):
            # GPU-safe auto balancing
            try: self.estimator.set_params(auto_class_weights="Balanced")
            except Exception: pass
            try: self.estimator.set_params(eval_metric=metric)
            except Exception: pass

        # Fit with ES if we have any validation (single-class OK with Logloss)
        has_valid = (Xva is not None and len(yva) > 0)
        if has_valid and self._is_xgb(self.estimator):
            import xgboost as xgb
            self.estimator.fit(
                Xtr, ytr,
                eval_set=[(Xva, yva)],
                verbose=False,
                callbacks=[xgb.callback.EarlyStopping(
                    rounds=int(patience),
                    metric_name=metric,
                    data_name="validation_0",
                    save_best=True
                )]
            )
        elif has_valid and self._is_catboost(self.estimator):
            from catboost import Pool
            self.estimator.set_params(
                use_best_model=True,
                od_type="Iter",
                od_wait=int(patience),
                custom_metric=["PRAUC:type=Classic;hints=skip_train~true"],
            )
            self.estimator.fit(
                Xtr, ytr,
                eval_set=Pool(Xva, yva),
                verbose=False,
                metric_period=50
            )
        else:
            # Fall back: train on train split without ES
            self.estimator.fit(Xtr, ytr)

        self.classes_ = getattr(self.estimator, "classes_", np.array([0, 1]))
        self._tr_idx_ = tr_idx; self._va_idx_ = va_idx; self._pos_rate_ = pos_rate
        return self

    def predict_proba(self, X: pd.DataFrame):
        return self.estimator.predict_proba(X)

    def predict(self, X: pd.DataFrame):
        return self.estimator.predict(X)

    # -------------------------- helpers --------------------------
    def _compute_train_val_indices(self, y: np.ndarray, n_total: int):
        rng = np.random.default_rng(self.random_state)
        n_classes = np.unique(y).size

        def full_data_split():
            if self.valid_size <= 0 or n_classes < 2:
                idx = rng.permutation(n_total); return idx, None
            sss = StratifiedShuffleSplit(n_splits=1, test_size=self.valid_size, random_state=self.random_state)
            tr, va = next(sss.split(np.zeros(n_total, dtype=np.int8), y))
            return tr, va

        if self.n_samples is None or self.n_samples >= n_total:
            return full_data_split()

        # Use n_samples for train; build val from remainder (capped)
        sss_tr = StratifiedShuffleSplit(n_splits=1, train_size=self.n_samples, random_state=self.random_state)
        tr_idx, rest_idx = next(sss_tr.split(np.zeros(n_total, dtype=np.int8), y))
        remaining = len(rest_idx)

        min_val_needed = int(np.ceil(self.n_samples * max(self.valid_size, 0.0)))
        val_cap = max(min_val_needed, int(round(self.val_cap_ratio * self.n_samples)))
        want_val = min(remaining, val_cap)

        y_rest = y[rest_idx]
        if remaining < min_val_needed or np.unique(y_rest).size < 2 or self.valid_size <= 0:
            return full_data_split()

        sss_val = StratifiedShuffleSplit(n_splits=1, train_size=want_val, random_state=self.random_state)
        try:
            va_sel, _ = next(sss_val.split(np.zeros(remaining, dtype=np.int8), y_rest))
        except ValueError:
            return full_data_split()

        va_idx = rest_idx[va_sel]
        return tr_idx, va_idx

    def _choose_metric(self, pos_rate=0.01) -> str:
        if self.es_metric != "auto":
            return self.es_metric
        if pos_rate is None or pos_rate == 0.0 or pos_rate == 1.0:
            return "logloss" if self._is_xgb(self.estimator) else "Logloss"
        return "aucpr" if self._is_xgb(self.estimator) else "PRAUC:type=Classic"

    def _choose_patience(self, pos_rate: Optional[float]) -> int:
        if isinstance(self.es_rounds, int):
            return self.es_rounds
        try:
            n_estimators = (int(self.estimator.get_params().get("n_estimators", 200))
                            if self._is_xgb(self.estimator)
                            else int(self.estimator.get_params().get("iterations", 500)))
        except Exception:
            n_estimators = 200
        base = max(30, int(round(0.20 * (n_estimators or 200))))
        if pos_rate is None:
            return base
        if pos_rate < 0.005:   # <0.5%
            return int(round(base * 1.75))
        if pos_rate < 0.02:    # <2%
            return int(round(base * 1.40))
        return base

    @staticmethod
    def _is_xgb(est):
        name = est.__class__.__name__.lower(); mod = getattr(est, "__module__", "")
        return "xgb" in name or "xgboost" in mod or hasattr(est, "get_xgb_params")

    @staticmethod
    def _is_catboost(est):
        name = est.__class__.__name__.lower(); mod = getattr(est, "__module__", "")
        return "catboost" in name or "catboost" in mod or hasattr(est, "get_all_params")


class StratifiedSubsetClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator, n_samples, random_state=SEED):
        self.estimator = estimator
        self.n_samples = n_samples and int(n_samples)
        self.random_state = random_state

    def fit(self, X, y):
        y = np.asarray(y)
        n_total = len(y)

        if self.n_samples is None or self.n_samples >= n_total:
            rng = np.random.default_rng(self.random_state)
            idx = rng.permutation(n_total)
        else:
            sss = StratifiedShuffleSplit(
                n_splits=1, train_size=self.n_samples, random_state=self.random_state
            )
            idx, _ = next(sss.split(np.zeros(n_total, dtype=np.int8), y))

        Xn = X.iloc[idx]
        Xn = Xn.to_numpy(np.float32, copy=False)
        yn = y[idx]

        self.estimator.fit(Xn, yn)
        self.classes_ = getattr(self.estimator, "classes_", np.array([0, 1]))
        return self

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

    def predict(self, X):
        return self.estimator.predict(X)


In [ ]:
# ==================== SCORING FUNCTIONS ====================

class HostVisibleError(Exception):
    pass

def single_lab_f1(lab_solution: pl.DataFrame, lab_submission: pl.DataFrame, beta: float = 1) -> float:
    label_frames: defaultdict[str, set[int]] = defaultdict(set)
    prediction_frames: defaultdict[str, set[int]] = defaultdict(set)

    for row in lab_solution.to_dicts():
        label_frames[row['label_key']].update(range(row['start_frame'], row['stop_frame']))

    for video in lab_solution['video_id'].unique():
        active_labels: str = lab_solution.filter(pl.col('video_id') == video)['behaviors_labeled'].first()
        active_labels: set[str] = set(json.loads(active_labels))
        predicted_mouse_pairs: defaultdict[str, set[int]] = defaultdict(set)

        for row in lab_submission.filter(pl.col('video_id') == video).to_dicts():
            if ','.join([str(row['agent_id']), str(row['target_id']), row['action']]) not in active_labels:
                continue
           
            new_frames = set(range(row['start_frame'], row['stop_frame']))
            new_frames = new_frames.difference(prediction_frames[row['prediction_key']])
            prediction_pair = ','.join([str(row['agent_id']), str(row['target_id'])])
            if predicted_mouse_pairs[prediction_pair].intersection(new_frames):
                raise HostVisibleError('Multiple predictions for the same frame from one agent/target pair')
            prediction_frames[row['prediction_key']].update(new_frames)
            predicted_mouse_pairs[prediction_pair].update(new_frames)

    tps = defaultdict(int)
    fns = defaultdict(int)
    fps = defaultdict(int)
    for key, pred_frames in prediction_frames.items():
        action = key.split('_')[-1]
        matched_label_frames = label_frames[key]
        tps[action] += len(pred_frames.intersection(matched_label_frames))
        fns[action] += len(matched_label_frames.difference(pred_frames))
        fps[action] += len(pred_frames.difference(matched_label_frames))

    distinct_actions = set()
    for key, frames in label_frames.items():
        action = key.split('_')[-1]
        distinct_actions.add(action)
        if key not in prediction_frames:
            fns[action] += len(frames)

    action_f1s = []
    for action in distinct_actions:
        if tps[action] + fns[action] + fps[action] == 0:
            action_f1s.append(0)
        else:
            action_f1s.append((1 + beta**2) * tps[action] / ((1 + beta**2) * tps[action] + beta**2 * fns[action] + fps[action]))
    return sum(action_f1s) / len(action_f1s)

def mouse_fbeta(solution: pd.DataFrame, submission: pd.DataFrame, beta: float = 1) -> float:
    if len(solution) == 0 or len(submission) == 0:
        raise ValueError('Missing solution or submission data')

    expected_cols = ['video_id', 'agent_id', 'target_id', 'action', 'start_frame', 'stop_frame']

    for col in expected_cols:
        if col not in solution.columns:
            raise ValueError(f'Solution is missing column {col}')
        if col not in submission.columns:
            raise ValueError(f'Submission is missing column {col}')

    solution: pl.DataFrame = pl.DataFrame(solution)
    submission: pl.DataFrame = pl.DataFrame(submission)
    assert (solution['start_frame'] <= solution['stop_frame']).all()
    assert (submission['start_frame'] <= submission['stop_frame']).all()
    solution_videos = set(solution['video_id'].unique())
    submission = submission.filter(pl.col('video_id').is_in(solution_videos))

    solution = solution.with_columns(
        pl.concat_str(
            [
                pl.col('video_id').cast(pl.Utf8),
                pl.col('agent_id').cast(pl.Utf8),
                pl.col('target_id').cast(pl.Utf8),
                pl.col('action'),
            ],
            separator='_',
        ).alias('label_key'),
    )
    submission = submission.with_columns(
        pl.concat_str(
            [
                pl.col('video_id').cast(pl.Utf8),
                pl.col('agent_id').cast(pl.Utf8),
                pl.col('target_id').cast(pl.Utf8),
                pl.col('action'),
            ],
            separator='_',
        ).alias('prediction_key'),
    )

    lab_scores = []
    for lab in solution['lab_id'].unique():
        lab_solution = solution.filter(pl.col('lab_id') == lab).clone()
        lab_videos = set(lab_solution['video_id'].unique())
        lab_submission = submission.filter(pl.col('video_id').is_in(lab_videos)).clone()
        lab_scores.append(single_lab_f1(lab_solution, lab_submission, beta=beta))

    return sum(lab_scores) / len(lab_scores)

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str, beta: float = 1) -> float:
    solution = solution.drop(row_id_column_name, axis='columns', errors='ignore')
    submission = submission.drop(row_id_column_name, axis='columns', errors='ignore')
    return mouse_fbeta(solution, submission, beta=beta)

In [ ]:
# ==================== TRACKING PRE-SMOOTH (optional) ====================
# 短欠損補間 + 小窓中央値平滑を切り替え可能にする。

ENABLE_TRACK_INTERP = os.environ.get("ENABLE_TRACK_INTERP", "true").lower() == "true"
ENABLE_TRACK_MEDIAN = os.environ.get("ENABLE_TRACK_MEDIAN", "true").lower() == "true"
ENABLE_TRACK_DEBUG = os.environ.get("ENABLE_TRACK_DEBUG", "true").lower() == "true"
TRACK_MAX_GAP = int(os.environ.get("TRACK_MAX_GAP", 3))          # 補間する最大連続欠損フレーム数
TRACK_MED_WIN = int(os.environ.get("TRACK_MED_WIN", 3))           # 中央値平滑窓（3 または 5 を想定）


def smooth_tracking(pvid: pd.DataFrame,
                    max_gap: int = TRACK_MAX_GAP,
                    med_win: int = TRACK_MED_WIN,
                    enable_interp: bool = ENABLE_TRACK_INTERP,
                    enable_median: bool = ENABLE_TRACK_MEDIAN,
                    debug: bool = ENABLE_TRACK_DEBUG) -> pd.DataFrame:
    """短欠損補間と小窓中央値平滑を適用する。
    - max_gap: limit を超える欠損は埋めない（長欠損の誤補間を防ぐ）。
    - med_win: 3 or 5 を推奨。2 以下なら平滑をスキップ。
    - enable_* で挙動を環境変数から切替可能。
    """
    if not (enable_interp or enable_median):
        return pvid

    df = pvid.copy()
    nan_before = df.isna().sum().sum() if debug else None

    for col in df.columns:
        s = df[col]
        if enable_interp:
            s = s.interpolate(limit=max_gap, limit_direction="both")
        if enable_median and (med_win is not None) and (med_win > 1):
            s = s.rolling(med_win, center=True, min_periods=1).median()
        df[col] = s

    df = df.astype(np.float32, copy=False)

    if debug:
        nan_after = df.isna().sum().sum()
        print(f"[smooth_tracking] nan {nan_before}->{nan_after} | interp={enable_interp} | med={enable_median} | win={med_win} | max_gap={max_gap}")
    return df



In [ ]:
# ==================== DATA LOADING ====================

train = pd.read_csv('/kaggle/input/MABe-mouse-behavior-detection/train.csv')

# config: optionally drop 25fps AdaptableSnail videos (labels suspected noisy)
EXCLUDE_ADAPT_25FPS = os.environ.get("EXCLUDE_ADAPT_25FPS", "true").lower() == "true"

# drop likely-sleeping MABe22 clips: condition == "lights on"
train = train.loc[~(train['lab_id'].astype(str).str.contains('MABe22', na=False) &
                    train['mouse1_condition'].astype(str).str.lower().eq('lights on'))].copy()

if EXCLUDE_ADAPT_25FPS:
    before = len(train)
    keep_mask = ~(
        train['lab_id'].astype(str).str.lower().str.contains('adaptablesnail', na=False)
        & np.isclose(train['frames_per_second'].astype(float), 25.0)
    )
    train = train.loc[keep_mask].copy()
    if verbose:
        print(f"drop AdaptableSnail 25fps train rows: {before - len(train)}")

train['n_mice'] = 4 - train[['mouse1_strain', 'mouse2_strain', 'mouse3_strain', 'mouse4_strain']].isna().sum(axis=1)

test = pd.read_csv('/kaggle/input/MABe-mouse-behavior-detection/test.csv')
test['sleeping'] = (
    test['lab_id'].astype(str).str.contains('MABe22', na=False) &
    test['mouse1_condition'].astype(str).str.lower().eq('lights on')
)
test['n_mice'] = 4 - test[['mouse1_strain','mouse2_strain','mouse3_strain','mouse4_strain']].isna().sum(axis=1)

# test が 1 行以下なら学習・推論をスキップするフラグ
SKIP_SMALL_TEST = len(test) <= 1
if SKIP_SMALL_TEST:
    print('test.csv が 1 行以下のため、学習/推論をスキップし submission.csv のみ出力します。')

body_parts_tracked_list = list(np.unique(train.body_parts_tracked))

drop_body_parts = ['headpiece_bottombackleft', 'headpiece_bottombackright', 'headpiece_bottomfrontleft', 'headpiece_bottomfrontright', 
                   'headpiece_topbackleft', 'headpiece_topbackright', 'headpiece_topfrontleft', 'headpiece_topfrontright',                  
                   'spine_1', 'spine_2', 'tail_middle_1', 'tail_middle_2', 'tail_midpoint']

_sex_cols = [f'mouse{i}_sex' for i in range(1,5)]
_train_sex_lut = (train[['video_id'] + _sex_cols].drop_duplicates('video_id')
                  .set_index('video_id').to_dict('index'))
_test_sex_lut  = (test[['video_id']  + _sex_cols].drop_duplicates('video_id')
                  .set_index('video_id').to_dict('index'))
_FEATURE_TEMPLATES = {}

def generate_mouse_data(dataset, traintest, traintest_directory=None,
                        generate_single=True, generate_pair=True):
    assert traintest in ['train', 'test']
    if traintest_directory is None:
        traintest_directory = f"/kaggle/input/MABe-mouse-behavior-detection/{traintest}_tracking"

    def _to_num(x):
        if isinstance(x, (int, np.integer)): return int(x)
        m = re.search(r'(\d+)$', str(x))
        return int(m.group(1)) if m else None

    for _, row in dataset.iterrows():
        lab_id   = row.lab_id
        video_id = row.video_id
        fps      = float(row.frames_per_second)
        n_mice   = int(row.n_mice)
        arena_w  = float(row.get('arena_width_cm', np.nan))
        arena_h  = float(row.get('arena_height_cm', np.nan))
        sleeping = bool(getattr(row, 'sleeping', False))
        arena_shape = row.get('arena_shape', 'rectangular')

        if not isinstance(row.behaviors_labeled, str):
            continue

        # ---- tracking ----
        path = f"{traintest_directory}/{lab_id}/{video_id}.parquet"
        vid = pd.read_parquet(path)
        if len(np.unique(vid.bodypart)) > 5:
            vid = vid.query("~ bodypart.isin(@drop_body_parts)")
        pvid = vid.pivot(columns=['mouse_id','bodypart'], index='video_frame', values=['x','y'])
        del vid
        pvid = pvid.reorder_levels([1,2,0], axis=1).T.sort_index().T
        pvid = (pvid / float(row.pix_per_cm_approx)).astype('float32', copy=False)

        if ENABLE_TRACK_INTERP or ENABLE_TRACK_MEDIAN:
            pvid = smooth_tracking(
                pvid,
                max_gap=TRACK_MAX_GAP,
                med_win=TRACK_MED_WIN,
                enable_interp=ENABLE_TRACK_INTERP,
                enable_median=ENABLE_TRACK_MEDIAN,
                debug=ENABLE_TRACK_DEBUG,
            )

        # available mouse_id labels in tracking (could be ints or strings)
        avail = list(pvid.columns.get_level_values('mouse_id').unique())
        avail_set = set(avail) | set(map(str, avail)) | {f"mouse{_to_num(a)}" for a in avail if _to_num(a) is not None}

        def _resolve(agent_str):
            """Return the matching mouse_id label present in pvid (int or str), or None."""
            m = re.search(r'(\d+)$', str(agent_str))
            cand = [agent_str]
            if m:
                n = int(m.group(1))
                cand = [n, n-1, str(n), f"mouse{n}", agent_str]  # try 1-based, 0-based, str, canonical
            for c in cand:
                if c in avail_set:  # compare within unified set
                    # return the exact label used in columns
                    if c in set(avail): return c
                    # map back to the exact label that exists (int preferred)
                    for a in avail:
                        if str(a) == str(c) or f"mouse{_to_num(a)}" == str(c):
                            return a
            return None

        # ---- behaviors ----
        vb = json.loads(row.behaviors_labeled)
        vb = sorted(list({b.replace("'", "") for b in vb}))
        vb = pd.DataFrame([b.split(',') for b in vb], columns=['agent','target','action'])
        vb['agent']  = vb['agent'].astype(str)
        vb['target'] = vb['target'].astype(str)
        vb['action'] = vb['action'].astype(str).str.lower()

        if traintest == 'train':
            try:
                annot = pd.read_parquet(path.replace('train_tracking', 'train_annotation'))
            except FileNotFoundError:
                continue

        def _mk_meta(index, agent_id, target_id):
            m = pd.DataFrame({
                'lab_id':        lab_id,
                'video_id':      video_id,
                'agent_id':      agent_id,
                'target_id':     target_id,
                'video_frame':   index.astype('int32', copy=False),
                'frames_per_second': np.float32(fps),
                'sleeping':      sleeping,
                'arena_shape':   arena_shape,
                'arena_width_cm': np.float32(arena_w),
                'arena_height_cm': np.float32(arena_h),
                'n_mice':        np.int8(n_mice),
            })
            for c in ('lab_id','video_id','agent_id','target_id','arena_shape'):
                m[c] = m[c].astype('category')
            return m

        # ---------- SINGLE ----------
        if generate_single:
            vb_single = vb.query("target == 'self'")
            for agent_str in pd.unique(vb_single['agent']):
                col_lab = _resolve(agent_str)
                if col_lab is None:
                    # if verbose: print(f"[skip single] {video_id} missing {agent_str} in tracking (avail={sorted(avail)})")
                    continue
                actions = sorted(vb_single.loc[vb_single['agent'].eq(agent_str), 'action'].unique().tolist())
                if not actions:
                    continue

                single = pvid.loc[:, col_lab]
                meta_df = _mk_meta(single.index, agent_str, 'self')

                if traintest == 'train':
                    a_num = _to_num(col_lab)
                    y = pd.DataFrame(False, index=single.index.astype('int32', copy=False), columns=actions)
                    a_sub = annot.query("(agent_id == @a_num) & (target_id == @a_num)")
                    for i in range(len(a_sub)):
                        ar = a_sub.iloc[i]
                        a = str(ar.action).lower()
                        if a in y.columns:
                            y.loc[int(ar['start_frame']):int(ar['stop_frame']), a] = True
                    yield 'single', single, meta_df, y
                else:
                    yield 'single', single, meta_df, actions

        # ---------- PAIR (ONLY LABELED PAIRS) ----------
        if generate_pair:
            vb_pair = vb.query("target != 'self'")
            if len(vb_pair) > 0:
                allowed_pairs = set(map(tuple, vb_pair[['agent','target']].itertuples(index=False, name=None)))

                for agent_num, target_num in itertools.permutations(
                        np.unique(pvid.columns.get_level_values('mouse_id')), 2):
                    agent_str = f"mouse{_to_num(agent_num)}"
                    target_str = f"mouse{_to_num(target_num)}"
                    if (agent_str, target_str) not in allowed_pairs:
                        continue

                    a_col = _resolve(agent_str)
                    b_col = _resolve(target_str)
                    if a_col is None or b_col is None:
                        # if verbose: print(f"[skip pair] {video_id} missing {agent_str}->{target_str}")
                        continue

                    actions = sorted(
                        vb_pair.query("(agent == @agent_str) & (target == @target_str)")['action'].unique().tolist()
                    )
                    if not actions:
                        continue

                    pair_xy = pd.concat([pvid[a_col], pvid[b_col]], axis=1, keys=['A','B'])
                    meta_df = _mk_meta(pair_xy.index, agent_str, target_str)

                    if traintest == 'train':
                        a_num = _to_num(a_col); b_num = _to_num(b_col)
                        y = pd.DataFrame(False, index=pair_xy.index.astype('int32', copy=False), columns=actions)
                        a_sub = annot.query("(agent_id == @a_num) & (target_id == @b_num)")
                        for i in range(len(a_sub)):
                            ar = a_sub.iloc[i]
                            a = str(ar.action).lower()
                            if a in y.columns:
                                y.loc[int(ar['start_frame']):int(ar['stop_frame']), a] = True
                        yield 'pair', pair_xy, meta_df, y
                    else:
                        yield 'pair', pair_xy, meta_df, actions


In [ ]:
# ==================== ADAPTIVE THRESHOLDING ====================

def predict_multiclass_adaptive(pred, meta, action_thresholds=None, smooth_windows=None, min_event_len=3, min_event_len_map=None):
    """Adaptive thresholding per action + temporal smoothing.
    - action_thresholds: dict(action->thr)
    - smooth_windows: dict(action->window) or None (default 5)
    - min_event_len_map: dict(action->min_len) or None (default min_event_len)
    """
    if action_thresholds is None:
        action_thresholds = defaultdict(lambda: 0.27)
    if smooth_windows is None:
        smooth_windows = {}
    if min_event_len_map is None:
        min_event_len_map = {}

    # Apply temporal smoothing (per action window if specified)
    pred_smoothed = pred.copy()
    for col in pred.columns:
        w = smooth_windows.get(col, 5)
        pred_smoothed[col] = pred[col].rolling(window=w, min_periods=1, center=True).mean()
    
    ama = np.argmax(pred_smoothed, axis=1)
    
    max_probs = pred_smoothed.max(axis=1)
    threshold_mask = np.zeros(len(pred_smoothed), dtype=bool)
    for i, action in enumerate(pred_smoothed.columns):
        action_mask = (ama == i)
        threshold = action_thresholds.get(action, 0.27)
        threshold_mask |= (action_mask & (max_probs >= threshold))
    
    ama = np.where(threshold_mask, ama, -1)
    ama = pd.Series(ama, index=meta.video_frame)
    
    changes_mask = (ama != ama.shift(1)).values
    ama_changes = ama[changes_mask]
    meta_changes = meta[changes_mask]
    mask = ama_changes.values >= 0
    mask[-1] = False
    
    submission_part = pd.DataFrame({
        'video_id': meta_changes['video_id'][mask].values,
        'agent_id': meta_changes['agent_id'][mask].values,
        'target_id': meta_changes['target_id'][mask].values,
        'action': pred.columns[ama_changes[mask].values],
        'start_frame': ama_changes.index[mask],
        'stop_frame': ama_changes.index[1:][mask[:-1]]
    })
    
    stop_video_id = meta_changes['video_id'][1:][mask[:-1]].values
    stop_agent_id = meta_changes['agent_id'][1:][mask[:-1]].values
    stop_target_id = meta_changes['target_id'][1:][mask[:-1]].values
    
    for i in range(len(submission_part)):
        video_id = submission_part.video_id.iloc[i]
        agent_id = submission_part.agent_id.iloc[i]
        target_id = submission_part.target_id.iloc[i]
        if i < len(stop_video_id):
            if stop_video_id[i] != video_id or stop_agent_id[i] != agent_id or stop_target_id[i] != target_id:
                new_stop_frame = meta.query("(video_id == @video_id)").video_frame.max() + 1
                submission_part.iat[i, submission_part.columns.get_loc('stop_frame')] = new_stop_frame
        else:
            new_stop_frame = meta.query("(video_id == @video_id)").video_frame.max() + 1
            submission_part.iat[i, submission_part.columns.get_loc('stop_frame')] = new_stop_frame
    
    # Filter out very short events (likely noise)
    duration = submission_part.stop_frame - submission_part.start_frame
    min_len_per_row = submission_part['action'].map(lambda a: min_event_len_map.get(a, min_event_len)).values
    submission_part = submission_part[duration.values >= min_len_per_row].reset_index(drop=True)
    
    if len(submission_part) > 0:
        assert (submission_part.stop_frame > submission_part.start_frame).all(), 'stop <= start'
    
    if verbose: print(f'  actions found: {len(submission_part)}')
    return submission_part

In [ ]:
# ==================== ADVANCED FEATURE ENGINEERING (FPS-AWARE) ====================

def safe_rolling(series, window, func, min_periods=None):
    """Safe rolling operation with NaN handling"""
    if min_periods is None:
        min_periods = max(1, window // 4)
    return series.rolling(window, min_periods=min_periods, center=True).apply(func, raw=True)

def _scale(n_frames_at_30fps, fps, ref=30.0):
    """Scale a frame count defined at 30 fps to the current video's fps."""
    return max(1, int(round(n_frames_at_30fps * float(fps) / ref)))

def _scale_signed(n_frames_at_30fps, fps, ref=30.0):
    """Signed version of _scale for forward/backward shifts (keeps at least 1 frame when |n|>=1)."""
    if n_frames_at_30fps == 0:
        return 0
    s = 1 if n_frames_at_30fps > 0 else -1
    mag = max(1, int(round(abs(n_frames_at_30fps) * float(fps) / ref)))
    return s * mag

def _fps_from_meta(meta_df, fallback_lookup, default_fps=30.0):
    if 'frames_per_second' in meta_df.columns and pd.notnull(meta_df['frames_per_second']).any():
        return float(meta_df['frames_per_second'].iloc[0])
    vid = meta_df['video_id'].iloc[0]
    return float(fallback_lookup.get(vid, default_fps))

def winsor_series(s: pd.Series, upper_q: float = 0.995, lower_q: float | None = None) -> pd.Series:
    """Clip extreme spikes by quantile; keeps NaN positions intact."""
    if len(s) == 0:
        return s
    lq = (1 - upper_q) if lower_q is None else lower_q
    low = s.quantile(lq) if lq and lq > 0 else None
    high = s.quantile(upper_q)
    return s.clip(lower=low, upper=high)

def _speed(cx: pd.Series, cy: pd.Series, fps: float) -> pd.Series:
    return np.hypot(cx.diff(), cy.diff()).fillna(0.0) * float(fps)

def _roll_future_mean(s: pd.Series, w: int, min_p: int = 1) -> pd.Series:
    # mean over [t, t+w-1]
    return s.iloc[::-1].rolling(w, min_periods=min_p).mean().iloc[::-1]

def _roll_future_var(s: pd.Series, w: int, min_p: int = 2) -> pd.Series:
    # var over [t, t+w-1]
    return s.iloc[::-1].rolling(w, min_periods=min_p).var().iloc[::-1]


def add_curvature_features(X, center_x, center_y, fps):
    """Trajectory curvature (window lengths scaled by fps)."""
    vel_x = center_x.diff()
    vel_y = center_y.diff()
    acc_x = vel_x.diff()
    acc_y = vel_y.diff()

    cross_prod = vel_x * acc_y - vel_y * acc_x
    vel_mag = np.sqrt(vel_x**2 + vel_y**2)
    curvature = np.abs(cross_prod) / (vel_mag**3 + 1e-6)  # invariant to time scaling

    for w in [30, 60]:
        ws = _scale(w, fps)
        X[f'curv_mean_{w}'] = curvature.rolling(ws, min_periods=max(1, ws // 6)).mean()

    angle = np.arctan2(vel_y, vel_x)
    angle_change = np.abs(angle.diff())
    w = 30
    ws = _scale(w, fps)
    X[f'turn_rate_{w}'] = angle_change.rolling(ws, min_periods=max(1, ws // 6)).sum()

    return X

def add_multiscale_features(X, center_x, center_y, fps):
    """Multi-scale temporal features (speed in cm/s; windows scaled by fps)."""
    # displacement per frame is already in cm (pix normalized earlier); convert to cm/s
    speed = np.sqrt(center_x.diff()**2 + center_y.diff()**2) * float(fps)
    speed = winsor_series(speed, upper_q=0.995)

    scales = [10, 40, 160]
    for scale in scales:
        ws = _scale(scale, fps)
        if len(speed) >= ws:
            X[f'sp_m{scale}'] = speed.rolling(ws, min_periods=max(1, ws // 4)).mean()
            X[f'sp_s{scale}'] = speed.rolling(ws, min_periods=max(1, ws // 4)).std()

    if len(scales) >= 2 and f'sp_m{scales[0]}' in X.columns and f'sp_m{scales[-1]}' in X.columns:
        X['sp_ratio'] = X[f'sp_m{scales[0]}'] / (X[f'sp_m{scales[-1]}'] + 1e-6)

    return X

def add_state_features(X, center_x, center_y, fps):
    """Behavioral state transitions; bins adjusted so semantics are fps-invariant."""
    speed = np.sqrt(center_x.diff()**2 + center_y.diff()**2) * float(fps)  # cm/s
    speed = winsor_series(speed, upper_q=0.995)
    w_ma = _scale(15, fps)
    speed_ma = speed.rolling(w_ma, min_periods=max(1, w_ma // 3)).mean()

    try:
        # Original bins (cm/frame): [-inf, 0.5, 2.0, 5.0, inf]
        # Convert to cm/s by multiplying by fps to keep thresholds consistent across fps.
        bins = [-np.inf, 0.5 * fps, 2.0 * fps, 5.0 * fps, np.inf]
        speed_states = pd.cut(speed_ma, bins=bins, labels=[0, 1, 2, 3]).astype(float)

        for window in [60, 120]:
            ws = _scale(window, fps)
            if len(speed_states) >= ws:
                for state in [0, 1, 2, 3]:
                    X[f's{state}_{window}'] = (
                        (speed_states == state).astype(float)
                        .rolling(ws, min_periods=max(1, ws // 6)).mean()
                    )
                state_changes = (speed_states != speed_states.shift(1)).astype(float)
                X[f'trans_{window}'] = state_changes.rolling(ws, min_periods=max(1, ws // 6)).sum()
    except Exception:
        pass

    return X

def add_longrange_features(X, center_x, center_y, fps):
    """Long-range temporal features (windows & spans scaled by fps)."""
    for window in [120, 240]:
        ws = _scale(window, fps)
        if len(center_x) >= ws:
            X[f'x_ml{window}'] = center_x.rolling(ws, min_periods=max(5, ws // 6)).mean()
            X[f'y_ml{window}'] = center_y.rolling(ws, min_periods=max(5, ws // 6)).mean()

    # EWM spans also interpreted in frames
    for span in [60, 120]:
        s = _scale(span, fps)
        X[f'x_e{span}'] = center_x.ewm(span=s, min_periods=1).mean()
        X[f'y_e{span}'] = center_y.ewm(span=s, min_periods=1).mean()

    speed = np.sqrt(center_x.diff()**2 + center_y.diff()**2) * float(fps)  # cm/s
    speed = winsor_series(speed, upper_q=0.995)
    for window in [60, 120]:
        ws = _scale(window, fps)
        if len(speed) >= ws:
            X[f'sp_pct{window}'] = speed.rolling(ws, min_periods=max(5, ws // 6)).rank(pct=True)

    return X

def add_cumulative_distance_single(X, cx, cy, fps, horizon_frames_base: int = 180, colname: str = "path_cum180"):
    L = max(1, _scale(horizon_frames_base, fps))  # frames
    # step length (cm per frame since coords are cm)
    step = np.hypot(cx.diff(), cy.diff())
    # centered rolling sum over ~2L+1 frames (acausal)
    path = step.rolling(2*L + 1, min_periods=max(5, L//6), center=True).sum()
    X[colname] = path.fillna(0.0).astype(np.float32)
    return X


def add_groom_microfeatures(X, df, fps):
    parts = df.columns.get_level_values(0)
    if 'body_center' not in parts or 'nose' not in parts:
        return X

    cx = df['body_center']['x']; cy = df['body_center']['y']
    nx = df['nose']['x']; ny = df['nose']['y']

    cs = (np.sqrt(cx.diff()**2 + cy.diff()**2) * float(fps)).fillna(0)
    ns = (np.sqrt(nx.diff()**2 + ny.diff()**2) * float(fps)).fillna(0)

    w30 = _scale(30, fps)
    X['head_body_decouple'] = (ns / (cs + 1e-3)).clip(0, 10).rolling(w30, min_periods=max(1, w30//3)).median()

    r = np.sqrt((nx - cx)**2 + (ny - cy)**2)
    X['nose_rad_std'] = r.rolling(w30, min_periods=max(1, w30//3)).std().fillna(0)

    if 'tail_base' in parts:
        ang = np.arctan2(df['nose']['y']-df['tail_base']['y'], df['nose']['x']-df['tail_base']['x'])
        dang = np.abs(ang.diff()).fillna(0)
        X['head_orient_jitter'] = dang.rolling(w30, min_periods=max(1, w30//3)).mean()

    return X


def add_interaction_features(X, mouse_pair, avail_A, avail_B, fps):
    """Social interaction features (windows scaled by fps)."""
    if 'body_center' not in avail_A or 'body_center' not in avail_B:
        return X

    rel_x = mouse_pair['A']['body_center']['x'] - mouse_pair['B']['body_center']['x']
    rel_y = mouse_pair['A']['body_center']['y'] - mouse_pair['B']['body_center']['y']
    rel_dist = np.sqrt(rel_x**2 + rel_y**2)
    rel_dist = winsor_series(rel_dist, upper_q=0.995)

    # per-frame velocities (cm/frame)
    A_vx = mouse_pair['A']['body_center']['x'].diff()
    A_vy = mouse_pair['A']['body_center']['y'].diff()
    B_vx = mouse_pair['B']['body_center']['x'].diff()
    B_vy = mouse_pair['B']['body_center']['y'].diff()

    A_lead = (A_vx * rel_x + A_vy * rel_y) / (np.sqrt(A_vx**2 + A_vy**2) * rel_dist + 1e-6)
    B_lead = (B_vx * (-rel_x) + B_vy * (-rel_y)) / (np.sqrt(B_vx**2 + B_vy**2) * rel_dist + 1e-6)

    for window in [30, 60]:
        ws = _scale(window, fps)
        X[f'A_ld{window}'] = A_lead.rolling(ws, min_periods=max(1, ws // 6)).mean()
        X[f'B_ld{window}'] = B_lead.rolling(ws, min_periods=max(1, ws // 6)).mean()

    approach = -rel_dist.diff()  # decreasing distance => positive approach
    chase = approach * B_lead
    w = 30
    ws = _scale(w, fps)
    X[f'chase_{w}'] = chase.rolling(ws, min_periods=max(1, ws // 6)).mean()

    for window in [60, 120]:
        ws = _scale(window, fps)
        A_sp = np.sqrt(A_vx**2 + A_vy**2)
        B_sp = np.sqrt(B_vx**2 + B_vy**2)
        X[f'sp_cor{window}'] = A_sp.rolling(ws, min_periods=max(1, ws // 6)).corr(B_sp)

    return X

# ===============================================================
# 1) Past–vs–Future speed asymmetry (acausal, continuous)
#    Δv = mean_future(speed) - mean_past(speed)
# ===============================================================
def add_speed_asymmetry_future_past_single(
    X: pd.DataFrame, cx: pd.Series, cy: pd.Series, fps: float,
    horizon_base: int = 30, agg: str = "mean"
) -> pd.DataFrame:
    w = max(3, _scale(horizon_base, fps))
    v = _speed(cx, cy, fps)
    v = winsor_series(v, upper_q=0.995)
    if agg == "median":
        v_past = v.rolling(w, min_periods=max(3, w//4), center=False).median()
        v_fut  = v.iloc[::-1].rolling(w, min_periods=max(3, w//4)).median().iloc[::-1]
    else:
        v_past = v.rolling(w, min_periods=max(3, w//4), center=False).mean()
        v_fut  = _roll_future_mean(v, w, min_p=max(3, w//4))
    X["spd_asym_1s"] = (v_fut - v_past).fillna(0.0)
    return X

# ===============================================================
# 2) Distribution shift (future vs past) via symmetric KL of
#    Gaussian fits on speed 
# ===============================================================
def add_gauss_shift_speed_future_past_single(
    X: pd.DataFrame, cx: pd.Series, cy: pd.Series, fps: float,
    window_base: int = 30, eps: float = 1e-6
) -> pd.DataFrame:
    w = max(5, _scale(window_base, fps))
    v = _speed(cx, cy, fps)
    v = winsor_series(v, upper_q=0.995)

    mu_p = v.rolling(w, min_periods=max(3, w//4)).mean()
    va_p = v.rolling(w, min_periods=max(3, w//4)).var().clip(lower=eps)

    mu_f = _roll_future_mean(v, w, min_p=max(3, w//4))
    va_f = _roll_future_var(v, w, min_p=max(3, w//4)).clip(lower=eps)

    # KL(Np||Nf) + KL(Nf||Np)
    kl_pf = 0.5 * ((va_p/va_f) + ((mu_f - mu_p)**2)/va_f - 1.0 + np.log(va_f/va_p))
    kl_fp = 0.5 * ((va_f/va_p) + ((mu_p - mu_f)**2)/va_p - 1.0 + np.log(va_p/va_f))
    X["spd_symkl_1s"] = (kl_pf + kl_fp).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X


In [ ]:
def transform_single(single_mouse, body_parts_tracked, fps, meta_df=None):
    """Enhanced single mouse transform (FPS-aware windows/lags; distances in cm)."""
    available_body_parts = single_mouse.columns.get_level_values(0)

    # Base distance features (squared distances across body parts)
    X = pd.DataFrame({
        f"{p1}+{p2}": np.square(single_mouse[p1] - single_mouse[p2]).sum(axis=1, skipna=False)
        for p1, p2 in itertools.combinations(body_parts_tracked, 2)
        if p1 in available_body_parts and p2 in available_body_parts
    })
    X = X.reindex(columns=[f"{p1}+{p2}" for p1, p2 in itertools.combinations(body_parts_tracked, 2)], copy=False)

    # Speed-like features via lagged displacements (duration-aware lag)
    if all(p in single_mouse.columns for p in ['ear_left', 'ear_right', 'tail_base']):
        lag = _scale(10, fps)
        shifted = single_mouse[['ear_left', 'ear_right', 'tail_base']].shift(lag)
        speeds = pd.DataFrame({
            'sp_lf': np.square(single_mouse['ear_left'] - shifted['ear_left']).sum(axis=1, skipna=False),
            'sp_rt': np.square(single_mouse['ear_right'] - shifted['ear_right']).sum(axis=1, skipna=False),
            'sp_lf2': np.square(single_mouse['ear_left'] - shifted['tail_base']).sum(axis=1, skipna=False),
            'sp_rt2': np.square(single_mouse['ear_right'] - shifted['tail_base']).sum(axis=1, skipna=False),
        })
        X = pd.concat([X, speeds], axis=1)

    if 'nose+tail_base' in X.columns and 'ear_left+ear_right' in X.columns:
        X['elong'] = X['nose+tail_base'] / (X['ear_left+ear_right'] + 1e-6)

    # Body angle (orientation)
    if all(p in available_body_parts for p in ['nose', 'body_center', 'tail_base']):
        v1 = single_mouse['nose'] - single_mouse['body_center']
        v2 = single_mouse['tail_base'] - single_mouse['body_center']
        X['body_ang'] = (v1['x'] * v2['x'] + v1['y'] * v2['y']) / (
            np.sqrt(v1['x']**2 + v1['y']**2) * np.sqrt(v2['x']**2 + v2['y']**2) + 1e-6)

    # Core temporal features (windows scaled by fps)
    if 'body_center' in available_body_parts:
        cx = single_mouse['body_center']['x']
        cy = single_mouse['body_center']['y']

        for w in [5, 15, 30, 60]:
            ws = _scale(w, fps)
            roll = dict(min_periods=1, center=True)
            X[f'cx_m{w}'] = cx.rolling(ws, **roll).mean()
            X[f'cy_m{w}'] = cy.rolling(ws, **roll).mean()
            X[f'cx_s{w}'] = cx.rolling(ws, **roll).std()
            X[f'cy_s{w}'] = cy.rolling(ws, **roll).std()
            X[f'x_rng{w}'] = cx.rolling(ws, **roll).max() - cx.rolling(ws, **roll).min()
            X[f'y_rng{w}'] = cy.rolling(ws, **roll).max() - cy.rolling(ws, **roll).min()
            X[f'disp{w}'] = np.sqrt(cx.diff().rolling(ws, min_periods=1).sum()**2 +
                                     cy.diff().rolling(ws, min_periods=1).sum()**2)
            X[f'act{w}'] = np.sqrt(cx.diff().rolling(ws, min_periods=1).var() +
                                   cy.diff().rolling(ws, min_periods=1).var())

        # Advanced features (fps-scaled)
        X = add_curvature_features(X, cx, cy, fps)
        X = add_angular_velocity_features(X, cx, cy, fps)
        X = add_multiscale_features(X, cx, cy, fps)
        X = add_state_features(X, cx, cy, fps)
        X = add_longrange_features(X, cx, cy, fps)
        X = add_cumulative_distance_single(X, cx, cy, fps, horizon_frames_base=180)
        X = add_groom_microfeatures(X, single_mouse, fps)
        X = add_speed_asymmetry_future_past_single(X, cx, cy, fps, horizon_base=30)         
        X = add_gauss_shift_speed_future_past_single(X, cx, cy, fps, window_base=30)
  
    # Nose-tail features with duration-aware lags
    if all(p in available_body_parts for p in ['nose', 'tail_base']):
        nt_dist = np.sqrt((single_mouse['nose']['x'] - single_mouse['tail_base']['x'])**2 +
                          (single_mouse['nose']['y'] - single_mouse['tail_base']['y'])**2)
        for lag in [10, 20, 40]:
            l = _scale(lag, fps)
            X[f'nt_lg{lag}'] = nt_dist.shift(l)
            X[f'nt_df{lag}'] = nt_dist - nt_dist.shift(l)

    # Ear features with duration-aware offsets
    if all(p in available_body_parts for p in ['ear_left', 'ear_right']):
        ear_d = np.sqrt((single_mouse['ear_left']['x'] - single_mouse['ear_right']['x'])**2 +
                        (single_mouse['ear_left']['y'] - single_mouse['ear_right']['y'])**2)
        for off in [-20, -10, 10, 20]:
            o = _scale_signed(off, fps)
            X[f'ear_o{off}'] = ear_d.shift(-o)  
        w = _scale(30, fps)
        X['ear_con'] = ear_d.rolling(w, min_periods=1, center=True).std() / \
                       (ear_d.rolling(w, min_periods=1, center=True).mean() + 1e-6)

    # --- Environment meta features (only) ---
    X['fps'] = np.float32(fps)
    X['arena_w'] = np.float32(0.0)
    X['density'] = np.float32(0.0)
    try:
        if meta_df is not None:
            w_val = float(meta_df['arena_width_cm'].iloc[0]) if 'arena_width_cm' in meta_df.columns else np.nan
            h_val = float(meta_df['arena_height_cm'].iloc[0]) if 'arena_height_cm' in meta_df.columns else np.nan
            n_mice = float(meta_df['n_mice'].iloc[0]) if 'n_mice' in meta_df.columns else 2.0
            if np.isfinite(w_val) and np.isfinite(h_val) and (w_val > 1e-3) and (h_val > 1e-3):
                X['arena_w'] = np.float32(w_val)
                X['density'] = np.float32(n_mice / (w_val * h_val + 1e-6))
    except Exception:
        pass

    return X.astype(np.float32, copy=False)

def transform_pair(mouse_pair, body_parts_tracked, fps, meta_df=None):
    """Enhanced pair transform (FPS-aware windows/lags; distances in cm)."""
    avail_A = mouse_pair['A'].columns.get_level_values(0)
    avail_B = mouse_pair['B'].columns.get_level_values(0)

    # Inter-mouse distances (squared distances across all part pairs)
    X = pd.DataFrame({
        f"12+{p1}+{p2}": np.square(mouse_pair['A'][p1] - mouse_pair['B'][p2]).sum(axis=1, skipna=False)
        for p1, p2 in itertools.product(body_parts_tracked, repeat=2)
        if p1 in avail_A and p2 in avail_B
    })
    X = X.reindex(columns=[f"12+{p1}+{p2}" for p1, p2 in itertools.product(body_parts_tracked, repeat=2)], copy=False)

    # Speed-like features via lagged displacements (duration-aware lag)
    if ('A', 'ear_left') in mouse_pair.columns and ('B', 'ear_left') in mouse_pair.columns:
        lag = _scale(10, fps)
        shA = mouse_pair['A']['ear_left'].shift(lag)
        shB = mouse_pair['B']['ear_left'].shift(lag)
        speeds = pd.DataFrame({
            'sp_A': np.square(mouse_pair['A']['ear_left'] - shA).sum(axis=1, skipna=False),
            'sp_AB': np.square(mouse_pair['A']['ear_left'] - shB).sum(axis=1, skipna=False),
            'sp_B': np.square(mouse_pair['B']['ear_left'] - shB).sum(axis=1, skipna=False),
        })
        X = pd.concat([X, speeds], axis=1)

    if 'nose+tail_base' in X.columns and 'ear_left+ear_right' in X.columns:
        X['elong'] = X['nose+tail_base'] / (X['ear_left+ear_right'] + 1e-6)

    # Relative orientation
    if all(p in avail_A for p in ['nose', 'tail_base']) and all(p in avail_B for p in ['nose', 'tail_base']):
        dir_A = mouse_pair['A']['nose'] - mouse_pair['A']['tail_base']
        dir_B = mouse_pair['B']['nose'] - mouse_pair['B']['tail_base']
        X['rel_ori'] = (dir_A['x'] * dir_B['x'] + dir_A['y'] * dir_B['y']) / (
            np.sqrt(dir_A['x']**2 + dir_A['y']**2) * np.sqrt(dir_B['x']**2 + dir_B['y']**2) + 1e-6)

    # Approach rate (duration-aware lag)
    if all(p in avail_A for p in ['nose']) and all(p in avail_B for p in ['nose']):
        cur = np.square(mouse_pair['A']['nose'] - mouse_pair['B']['nose']).sum(axis=1, skipna=False)
        lag = _scale(10, fps)
        shA_n = mouse_pair['A']['nose'].shift(lag)
        shB_n = mouse_pair['B']['nose'].shift(lag)
        past = np.square(shA_n - shB_n).sum(axis=1, skipna=False)
        X['appr'] = cur - past

    # Distance bins (cm; unchanged by fps)
    if 'body_center' in avail_A and 'body_center' in avail_B:
        cd = np.sqrt((mouse_pair['A']['body_center']['x'] - mouse_pair['B']['body_center']['x'])**2 +
                     (mouse_pair['A']['body_center']['y'] - mouse_pair['B']['body_center']['y'])**2)
        cd = winsor_series(cd, upper_q=0.995)
        X['v_cls'] = (cd < 5.0).astype(float)
        X['cls']   = ((cd >= 5.0) & (cd < 15.0)).astype(float)
        X['med']   = ((cd >= 15.0) & (cd < 30.0)).astype(float)
        X['far']   = (cd >= 30.0).astype(float)

    # Temporal interaction features (fps-adjusted windows)
    if 'body_center' in avail_A and 'body_center' in avail_B:
        cd_full = np.square(mouse_pair['A']['body_center'] - mouse_pair['B']['body_center']).sum(axis=1, skipna=False)
        cd_full = winsor_series(cd_full, upper_q=0.995)

        for w in [5, 15, 30, 60]:
            ws = _scale(w, fps)
            roll = dict(min_periods=1, center=True)
            X[f'd_m{w}']  = cd_full.rolling(ws, **roll).mean()
            X[f'd_s{w}']  = cd_full.rolling(ws, **roll).std()
            X[f'd_mn{w}'] = cd_full.rolling(ws, **roll).min()
            X[f'd_mx{w}'] = cd_full.rolling(ws, **roll).max()

            d_var = cd_full.rolling(ws, **roll).var()
            X[f'int{w}'] = 1 / (1 + d_var)

            Axd = mouse_pair['A']['body_center']['x'].diff()
            Ayd = mouse_pair['A']['body_center']['y'].diff()
            Bxd = mouse_pair['B']['body_center']['x'].diff()
            Byd = mouse_pair['B']['body_center']['y'].diff()
            coord = Axd * Bxd + Ayd * Byd
            X[f'co_m{w}'] = coord.rolling(ws, **roll).mean()
            X[f'co_s{w}'] = coord.rolling(ws, **roll).std()

    # Nose-nose dynamics (duration-aware lags)
    if 'nose' in avail_A and 'nose' in avail_B:
        nn = np.sqrt((mouse_pair['A']['nose']['x'] - mouse_pair['B']['nose']['x'])**2 +
                     (mouse_pair['A']['nose']['y'] - mouse_pair['B']['nose']['y'])**2)
        nn = winsor_series(nn, upper_q=0.995)
        for lag in [10, 20, 40]:
            l = _scale(lag, fps)
            X[f'nn_lg{lag}']  = nn.shift(l)
            X[f'nn_ch{lag}']  = nn - nn.shift(l)
            is_cl = (nn < 10.0).astype(float)
            X[f'cl_ps{lag}']  = is_cl.rolling(l, min_periods=1).mean()

    # Velocity alignment (duration-aware offsets)
    if 'body_center' in avail_A and 'body_center' in avail_B:
        Avx = mouse_pair['A']['body_center']['x'].diff()
        Avy = mouse_pair['A']['body_center']['y'].diff()
        Bvx = mouse_pair['B']['body_center']['x'].diff()
        Bvy = mouse_pair['B']['body_center']['y'].diff()
        val = (Avx * Bvx + Avy * Bvy) / (np.sqrt(Avx**2 + Avy**2) * np.sqrt(Bvx**2 + Bvy**2) + 1e-6)

        for off in [-20, -10, 0, 10, 20]:
            o = _scale_signed(off, fps)
            X[f'va_{off}'] = val.shift(-o)

        w = _scale(30, fps)
        X['int_con'] = cd_full.rolling(w, min_periods=1, center=True).std() / \
                       (cd_full.rolling(w, min_periods=1, center=True).mean() + 1e-6)

        # Advanced interaction (fps-adjusted internals)
        X = add_interaction_features(X, mouse_pair, avail_A, avail_B, fps)
        

    # --- Environment meta features (only) ---
    X['fps'] = np.float32(fps)
    X['arena_w'] = np.float32(0.0)
    X['density'] = np.float32(0.0)
    try:
        if meta_df is not None:
            w_val = float(meta_df['arena_width_cm'].iloc[0]) if 'arena_width_cm' in meta_df.columns else np.nan
            h_val = float(meta_df['arena_height_cm'].iloc[0]) if 'arena_height_cm' in meta_df.columns else np.nan
            n_mice = float(meta_df['n_mice'].iloc[0]) if 'n_mice' in meta_df.columns else 2.0
            if np.isfinite(w_val) and np.isfinite(h_val) and (w_val > 1e-3) and (h_val > 1e-3):
                X['arena_w'] = np.float32(w_val)
                X['density'] = np.float32(n_mice / (w_val * h_val + 1e-6))
    except Exception:
        pass

    return X.astype(np.float32, copy=False)

In [ ]:
def add_angular_velocity_features(X, center_x, center_y, fps):
    """Angular velocity features (rotation motion detection)."""
    # 既存パターン：速度ベクトル
    dx = center_x.diff().fillna(0.0)
    dy = center_y.diff().fillna(0.0)
    
    # 既存パターン：角度計算（arctan2）
    angle = np.arctan2(dy, dx)
    
    # 角速度 = 角度の変化率
    angle_diff = angle.diff().fillna(0.0)
    # 角度差を[-π, π]の範囲に正規化（360度をまたぐ場合に対応）
    angle_diff = np.arctan2(np.sin(angle_diff), np.cos(angle_diff))
    angular_velocity = np.abs(angle_diff) * float(fps)  # rad/s
    
    # 既存パターン：winsorization（外れ値処理）
    angular_velocity = winsor_series(angular_velocity, upper_q=0.995)
    
    # 既存パターン：異常値クリップ
    angular_velocity = angular_velocity.clip(upper=10.0 * np.pi)
    
    # 既存パターン：複数ウィンドウでの統計量
    for w in [15, 30, 60]:
        ws = _scale(w, fps)
        roll = dict(min_periods=max(3, ws//4), center=True)
        X[f'ang_vel_mean_{w}'] = angular_velocity.rolling(ws, **roll).mean()
        X[f'ang_vel_std_{w}'] = angular_velocity.rolling(ws, **roll).std()
        X[f'ang_vel_max_{w}'] = angular_velocity.rolling(ws, **roll).max()
    
    return X



In [ ]:
# helpers
def _find_lgbm_step(pipe):
    try:
        if "stratifiedsubsetclassifier__estimator" in pipe.get_params():
            est = pipe.get_params()["stratifiedsubsetclassifier__estimator"]
            if isinstance(est, lightgbm.LGBMClassifier):
                return "stratifiedsubsetclassifier"
        if "stratifiedsubsetclassifierweval__estimator" in pipe.get_params():
            est = pipe.get_params()["stratifiedsubsetclassifierweval__estimator"]
            if isinstance(est, lightgbm.LGBMClassifier):
                return "stratifiedsubsetclassifierweval"
    except Exception as e:
        print(e)
    return None


def _short_hash(text: str, length: int = 10) -> str:
    return hashlib.md5(str(text).encode("utf-8")).hexdigest()[:length]


In [ ]:
def submit_ensemble(body_parts_tracked_str, switch_tr, X_tr, label, meta, n_samples=1_500_000):
    # CV 用と同じモデル構成を再利用（RF を含む）。二重管理を避けるため _build_models_with_rf を呼ぶ。
    models, model_names, factions = _build_models_with_rf(n_samples=n_samples)
    # Optuna 最適パラメタ（submit 用）を読み込む
    OPTUNA_BEST_PATH = os.environ.get("OPTUNA_BEST_PATH", "optuna_best.json")
    try:
        with open(OPTUNA_BEST_PATH, "r") as f:
            optuna_best = json.load(f)
    except Exception:
        optuna_best = {}

    model_list = []
    for action in label.columns:
        action_mask = ~label[action].isna().values
        y_action = label[action][action_mask].values.astype(int)
        meta_masked = meta.iloc[action_mask]

        # レアクラス全滅: 正例が少ない場合は学習せず、テスト時は0固定にする
        has_positive = (y_action.sum() > 1)  # StratifiedShuffleSplit が動く最小限を確保（>=2）
        trained = []
        trained_indices = []
        if has_positive:
            for model_idx, m in enumerate(models):
                m_clone = clone(m)
                try:
                    t0 = perf_counter()
                    m_clone.fit(X_tr[action_mask], y_action)
                    dt = perf_counter() - t0
                    print(f"trained model {model_names[model_idx]} | {switch_tr} | action={action} | {dt:.1f}s", flush=True)
                except Exception:
                    step = _find_lgbm_step(m_clone)
                    if step is None:
                        continue
                    try:
                        m_clone.set_params(**{f"{step}__estimator__device": "cpu"})
                        t0 = perf_counter()
                        m_clone.fit(X_tr[action_mask], y_action)
                        dt = perf_counter() - t0
                        print(f"trained (CPU fallback) {model_names[model_idx]} | {switch_tr} | action={action} | {dt:.1f}s", flush=True)
                    except Exception as e2:
                        print(e2)
                        continue
                trained.append(m_clone)
                trained_indices.append(model_idx)

        if trained:
            model_list.append((action, trained, trained_indices))
        else:
            # 学習できなかったクラスはダミー扱い（test では0を出す）
            model_list.append((action, [], []))

    del X_tr; gc.collect()

    # ---- TEST INFERENCE ----
    body_parts_tracked = json.loads(body_parts_tracked_str)
    if len(body_parts_tracked) > 5:
        body_parts_tracked = [b for b in body_parts_tracked if b not in drop_body_parts]

    test_subset = test[test.body_parts_tracked == body_parts_tracked_str]
    generator = generate_mouse_data(
        test_subset, 'test',
        generate_single=(switch_tr == 'single'),
        generate_pair=(switch_tr == 'pair')
    )
    fps_lookup = (test_subset[['video_id','frames_per_second']]
                    .drop_duplicates('video_id')
                    .set_index('video_id')['frames_per_second'].to_dict())

    for switch_te, data_te, meta_te, actions_te in generator:
        assert switch_te == switch_tr
        try:
            fps_i = _fps_from_meta(meta_te, fps_lookup, default_fps=30.0)
            if switch_te == 'single':
                X_te = transform_single(data_te, body_parts_tracked, fps_i, meta_df=meta_te)
            else:
                X_te = transform_pair(data_te, body_parts_tracked, fps_i, meta_df=meta_te)

            del data_te

            pred = pd.DataFrame(index=meta_te.video_frame)
            for action, trained, trained_indices_action in model_list:
                if action in actions_te:
                    if len(trained) == 0:
                        # 学習スキップしたクラスは0固定
                        pred[action] = np.zeros(len(meta_te), dtype=np.float32)
                        continue
                    all_probs = [mdl.predict_proba(X_te)[:, 1] for mdl in trained]
                    params = optuna_best.get(action, {})
                    w = params.get("w")
                    if w is not None and len(w) == 4:
                        # 派閥重みを適用（成功したモデルの元インデックスで整合を取る）
                        faction_means = {
                            "pred_lgbm": _avg_probs_by_faction(all_probs, trained_indices_action, factions["lgbm"]),
                            "pred_xgb": _avg_probs_by_faction(all_probs, trained_indices_action, factions["xgb"]),
                            "pred_cat": _avg_probs_by_faction(all_probs, trained_indices_action, factions["cat"]),
                            "pred_rf":  _avg_probs_by_faction(all_probs, trained_indices_action, factions["rf"]),
                        }
                        num = None; denom = 0.0
                        for key, weight in zip(["pred_lgbm","pred_xgb","pred_cat","pred_rf"], w):
                            if faction_means.get(key) is None or weight <= 0:
                                continue
                            num = faction_means[key] * weight if num is None else num + faction_means[key] * weight
                            denom += weight
                        if num is not None and denom > 0:
                            pred[action] = num / denom
                        else:
                            pred[action] = np.mean(all_probs, axis=0)
                    else:
                        # デフォルトは等重み平均
                        pred[action] = np.mean(all_probs, axis=0)

            del X_te; gc.collect()

            if pred.shape[1] != 0:
                # Optuna パラメタがあれば適用
                thr_map = {a: (optuna_best.get(a, {}).get("threshold", 0.27)) for a in pred.columns}
                smooth_map = {a: optuna_best.get(a, {}).get("smooth_w", 5) for a in pred.columns}
                minlen_map = {a: optuna_best.get(a, {}).get("min_event_len", 3) for a in pred.columns}
                submission_list.append(
                    predict_multiclass_adaptive(
                        pred,
                        meta_te,
                        action_thresholds=thr_map,
                        smooth_windows=smooth_map,
                        min_event_len=3,
                        min_event_len_map=minlen_map,
                    )
                )
        except Exception as e:
            print(e)
            try: del data_te
            except: pass
            gc.collect()


In [ ]:
# ==================== OOF CV & Optuna 用ユーティリティ ====================
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
import joblib

# 分散実行用パラメータ（環境変数で上書き可能）
N_SPLITS = int(os.environ.get("N_SPLITS", 3))
_MY_FOLD_ENV = os.environ.get("MY_FOLD", "")
MY_FOLD = None if _MY_FOLD_ENV in ("", "None", "-1") else int(_MY_FOLD_ENV)


def _ensure_dir(path_str: str) -> Path:
    path = Path(path_str)
    path.mkdir(parents=True, exist_ok=True)
    return path


def _build_models_with_rf(n_samples=1_500_000):
    """元のモデル構成に RF を加え、派閥別インデックスを返す。"""
    models = []
    model_names = []
    factions = {"lgbm": [], "xgb": [], "cat": [], "rf": []}

    models.append(make_pipeline(
        StratifiedSubsetClassifier(_make_lgbm(
            n_estimators=225, learning_rate=0.07, min_child_samples=40,
            num_leaves=31, subsample=0.8, colsample_bytree=0.8, verbose=-1, gpu_use_dp=USE_GPU
        ), n_samples)
    ))
    model_names.append("lgbm_225"); factions["lgbm"].append(len(models) - 1)

    models.append(make_pipeline(
        StratifiedSubsetClassifier(_make_lgbm(
            n_estimators=150, learning_rate=0.1, min_child_samples=20,
            num_leaves=63, max_depth=8, subsample=0.7, colsample_bytree=0.9,
            reg_alpha=0.1, reg_lambda=0.1, verbose=-1, gpu_use_dp=USE_GPU
        ), (n_samples and int(n_samples/1.25)))
    ))
    model_names.append("lgbm_150"); factions["lgbm"].append(len(models) - 1)

    models.append(make_pipeline(
        StratifiedSubsetClassifier(_make_lgbm(
            n_estimators=100, learning_rate=0.05, min_child_samples=30,
            num_leaves=127, max_depth=10, subsample=0.75, verbose=-1, gpu_use_dp=USE_GPU,
        ), (n_samples and int(n_samples/1.66)))
    ))
    model_names.append("lgbm_100"); factions["lgbm"].append(len(models) - 1)

    xgb0 = _make_xgb(
        n_estimators=180, learning_rate=0.08, max_depth=6,
        min_child_weight=8 if USE_GPU else 5, gamma=1.0 if USE_GPU else 0.,
        subsample=0.8, colsample_bytree=0.8, single_precision_histogram=USE_GPU,
        verbosity=0
    )
    models.append(make_pipeline(StratifiedSubsetClassifier(xgb0, n_samples and int(n_samples/1.2))))
    model_names.append("xgb_180"); factions["xgb"].append(len(models) - 1)

    cb_est = _make_cb(iterations=120, learning_rate=0.1, depth=6,
                      verbose=False, allow_writing_files=False)
    models.append(make_pipeline(StratifiedSubsetClassifier(cb_est, n_samples)))
    model_names.append("cat_120"); factions["cat"].append(len(models) - 1)

    if USE_GPU:
        xgb1 = XGBClassifier(
            random_state=SEED, booster="gbtree", tree_method="gpu_hist",
            n_estimators=2000, learning_rate=0.05, grow_policy="lossguide",
            max_leaves=255, max_depth=0, min_child_weight=10, gamma=0.0,
            subsample=0.90, colsample_bytree=1.00, colsample_bylevel=0.85,
            reg_alpha=0.0, reg_lambda=1.0, max_bin=256,
            single_precision_histogram=True, verbosity=0
        )
        models.append(make_pipeline(
            StratifiedSubsetClassifierWEval(xgb1, n_samples and int(n_samples/2.),
                                            random_state=SEED, valid_size=0.10, val_cap_ratio=0.25,
                                            es_rounds="auto", es_metric="auto")
        ))
        model_names.append("xgb1"); factions["xgb"].append(len(models) - 1)

        xgb2 = XGBClassifier(
            random_state=SEED, booster="gbtree", tree_method="gpu_hist",
            n_estimators=1400, learning_rate=0.06, max_depth=7,
            min_child_weight=12, subsample=0.70, colsample_bytree=0.80,
            reg_alpha=0.0, reg_lambda=1.5, max_bin=256,
            single_precision_histogram=True, verbosity=0
        )
        models.append(make_pipeline(
            StratifiedSubsetClassifierWEval(xgb2, n_samples and int(n_samples/1.5),
                                            random_state=SEED, valid_size=0.10, val_cap_ratio=0.25,
                                            es_rounds="auto", es_metric="auto")
        ))
        model_names.append("xgb2"); factions["xgb"].append(len(models) - 1)

        cb1 = CatBoostClassifier(
            random_seed=SEED, task_type="GPU", devices="0",
            iterations=4000, learning_rate=0.03, depth=8, l2_leaf_reg=6.0,
            bootstrap_type="Bayesian", bagging_temperature=0.5,
            random_strength=0.5, loss_function="Logloss",
            eval_metric="PRAUC:type=Classic", auto_class_weights="Balanced",
            border_count=64, verbose=False, allow_writing_files=False
        )
        models.append(make_pipeline(
            StratifiedSubsetClassifierWEval(cb1, n_samples and int(n_samples/2.0),
                                            random_state=SEED, valid_size=0.10, val_cap_ratio=0.25,
                                            es_rounds="auto", es_metric="auto")
        ))
        model_names.append("cat_bay"); factions["cat"].append(len(models) - 1)

    # XGBoost RF (派閥: rf) — 欠損値もそのまま扱える
    """
    rf_params = dict(
        random_state=SEED,
        tree_method="gpu_hist" if USE_GPU else "hist",
        subsample=0.8,
        colsample_bynode=0.8,
        n_estimators=400,
        learning_rate=0.1,
        max_depth=8,
        min_child_weight=5,
        reg_lambda=1.0,
        reg_alpha=0.0,
        n_jobs=-1,
        objective="binary:logistic",
    )
    rf_est = XGBRFClassifier(**rf_params)
    models.append(make_pipeline(StratifiedSubsetClassifier(rf_est, n_samples)))
    model_names.append("xgbrf_400"); factions["rf"].append(len(models) - 1)
    """
    return models, model_names, factions


def _avg_probs_by_faction(all_probs, trained_indices, idx_list):
    sel = [all_probs[i] for i, original_idx in enumerate(trained_indices) if original_idx in idx_list]
    if len(sel) == 0:
        return None
    return np.mean(sel, axis=0)


def submit_ensemble_cv(body_parts_tracked_str, switch_tr, X_tr, label, meta,
                       X_te=None, meta_te=None, actions_te=None,
                       n_samples=1_500_000, n_splits=N_SPLITS, my_fold=MY_FOLD,
                       save_dir="oof", pred_dir="pred"):
    """OOF 生成 + (任意)テスト予測保存。Optuna 前処理用。
    - 保存ファイル: oof_{hash}_{switch}_{action}_fold{fold}.pkl
                  pred_{hash}_{switch}_{action}_fold{fold}.pkl
    - OOF には派閥別列 pred_lgbm/pred_xgb/pred_cat/pred_rf を保存。
    """
    save_dir = _ensure_dir(save_dir)
    pred_dir = _ensure_dir(pred_dir)
    section_id = _short_hash(body_parts_tracked_str)

    def _oof_path(action, fold_idx):
        return save_dir / f"oof_{section_id}_{switch_tr}_{action}_fold{fold_idx}.pkl"

    def _pred_path(action, fold_idx):
        return pred_dir / f"pred_{section_id}_{switch_tr}_{action}_fold{fold_idx}.pkl"

    models, model_names, factions = _build_models_with_rf(n_samples=n_samples)

    for action in label.columns:
        mask = ~label[action].isna().values
        if mask.sum() == 0:
            continue

        y_action = label[action][mask].values.astype(int)
        pos_total = int(y_action.sum())
        if pos_total < n_splits:
            # 正例が n_splits 未満 → CV 全体をスキップし 0 埋めを保存
            zero_oof = {
                "video_id": meta.iloc[mask]["video_id"].values,
                "video_frame": meta.iloc[mask]["video_frame"].values,
                "y_true": y_action,
            }
            for k in ["pred_lgbm", "pred_xgb", "pred_cat", "pred_rf"]:
                zero_oof[k] = np.zeros_like(y_action, dtype=np.float32)
            joblib.dump(zero_oof, _oof_path(action, 0))

            if X_te is not None and meta_te is not None and ((actions_te is None) or (action in actions_te)):
                pred_zero = {
                    "video_id": meta_te["video_id"].values,
                    "video_frame": meta_te["video_frame"].values,
                }
                for k in ["pred_lgbm", "pred_xgb", "pred_cat", "pred_rf"]:
                    pred_zero[k] = np.zeros(len(meta_te), dtype=np.float32)
                joblib.dump(pred_zero, _pred_path(action, 0))
            continue

        X_action = X_tr.iloc[mask]
        meta_action = meta.iloc[mask]
        groups = meta_action["video_id"].astype(str).values

        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_action, y_action, groups)):
            if my_fold is not None and fold_idx != my_fold:
                continue

            y_tr_fold = y_action[train_idx]
            y_val_fold = y_action[val_idx]
            if (y_tr_fold.sum() < 2) or (y_val_fold.sum() == 0):
                # 正例なし: 学習せず OOF/pred を0埋めで出力
                oof_data = {
                    "video_id": meta_action.iloc[val_idx]["video_id"].values,
                    "video_frame": meta_action.iloc[val_idx]["video_frame"].values,
                    "y_true": y_val_fold,
                }
                for k in ["pred_lgbm", "pred_xgb", "pred_cat", "pred_rf"]:
                    oof_data[k] = np.zeros(len(y_val_fold), dtype=np.float32)
                joblib.dump(oof_data, _oof_path(action, fold_idx))

                if X_te is not None and meta_te is not None and ((actions_te is None) or (action in actions_te)):
                    pred_data = {
                        "video_id": meta_te["video_id"].values,
                        "video_frame": meta_te["video_frame"].values,
                    }
                    for k in ["pred_lgbm", "pred_xgb", "pred_cat", "pred_rf"]:
                        pred_data[k] = np.zeros(len(meta_te), dtype=np.float32)
                    joblib.dump(pred_data, _pred_path(action, fold_idx))
                continue

            trained_models = []
            trained_indices = []
            for model_idx, m in enumerate(models):
                m_clone = clone(m)
                try:
                    m_clone.fit(X_action.iloc[train_idx], y_tr_fold)
                    print(f"[OOF] trained {model_names[model_idx]} | {switch_tr} | action={action} | fold={fold_idx}", flush=True)
                except Exception:
                    step = _find_lgbm_step(m_clone)
                    if step is None:
                        continue
                    try:
                        m_clone.set_params(**{f"{step}__estimator__device": "cpu"})
                        m_clone.fit(X_action.iloc[train_idx], y_tr_fold)
                        print(f"[OOF] trained (CPU fallback) {model_names[model_idx]} | {switch_tr} | action={action} | fold={fold_idx}", flush=True)
                    except Exception as e2:
                        print(e2)
                        continue
                trained_models.append(m_clone)
                trained_indices.append(model_idx)

            if len(trained_models) == 0:
                print(f"[OOF] skip action={action} fold={fold_idx} (no model trained)")
                # OOF だけ 0 埋め
                oof_data = {
                    "video_id": meta_action.iloc[val_idx]["video_id"].values,
                    "video_frame": meta_action.iloc[val_idx]["video_frame"].values,
                    "y_true": y_val_fold,
                }
                for k in ["pred_lgbm", "pred_xgb", "pred_cat", "pred_rf"]:
                    oof_data[k] = np.zeros(len(y_val_fold), dtype=np.float32)
                joblib.dump(oof_data, _oof_path(action, fold_idx))
                continue

            val_probs = [mdl.predict_proba(X_action.iloc[val_idx])[:, 1] for mdl in trained_models]
            pred_lgbm = _avg_probs_by_faction(val_probs, trained_indices, factions["lgbm"])
            pred_xgb = _avg_probs_by_faction(val_probs, trained_indices, factions["xgb"])
            pred_cat = _avg_probs_by_faction(val_probs, trained_indices, factions["cat"])
            pred_rf  = _avg_probs_by_faction(val_probs, trained_indices, factions["rf"])

            oof_data = {
                "video_id": meta_action.iloc[val_idx]["video_id"].values,
                "video_frame": meta_action.iloc[val_idx]["video_frame"].values,
                "y_true": y_val_fold,
            }
            if pred_lgbm is not None: oof_data["pred_lgbm"] = pred_lgbm.astype(np.float32)
            if pred_xgb  is not None: oof_data["pred_xgb"]  = pred_xgb.astype(np.float32)
            if pred_cat  is not None: oof_data["pred_cat"]  = pred_cat.astype(np.float32)
            if pred_rf   is not None: oof_data["pred_rf"]   = pred_rf.astype(np.float32)

            joblib.dump(oof_data, _oof_path(action, fold_idx))

            if X_te is not None and meta_te is not None:
                if (actions_te is None) or (action in actions_te):
                    test_probs = [mdl.predict_proba(X_te)[:, 1] for mdl in trained_models]
                    t_lgbm = _avg_probs_by_faction(test_probs, trained_indices, factions["lgbm"])
                    t_xgb  = _avg_probs_by_faction(test_probs, trained_indices, factions["xgb"])
                    t_cat  = _avg_probs_by_faction(test_probs, trained_indices, factions["cat"])
                    t_rf   = _avg_probs_by_faction(test_probs, trained_indices, factions["rf"])
                    pred_data = {
                        "video_id": meta_te["video_id"].values,
                        "video_frame": meta_te["video_frame"].values,
                    }
                    if t_lgbm is not None: pred_data["pred_lgbm"] = t_lgbm.astype(np.float32)
                    if t_xgb  is not None: pred_data["pred_xgb"]  = t_xgb.astype(np.float32)
                    if t_cat  is not None: pred_data["pred_cat"]  = t_cat.astype(np.float32)
                    if t_rf   is not None: pred_data["pred_rf"]   = t_rf.astype(np.float32)
                    joblib.dump(pred_data, _pred_path(action, fold_idx))

    return True



In [ ]:
# ==================== Optuna 用後処理ユーティリティ ====================
import optuna
from sklearn.metrics import f1_score
from glob import glob

# 平滑化: video_id ごとに center=True rolling

def smooth_by_video(prob_df: pd.DataFrame, video_id: pd.Series, window: int) -> pd.DataFrame:
    if window <= 1:
        return prob_df
    out = []
    for vid, g in prob_df.assign(video_id=video_id.values).groupby("video_id"):
        rolled = g.drop(columns="video_id").rolling(window=window, min_periods=1, center=True).mean()
        out.append(rolled)
    return pd.concat(out, axis=0).sort_index()


def _remove_short_ones(arr: np.ndarray, min_len: int) -> np.ndarray:
    if min_len <= 1:
        return arr
    x = arr.astype(int)
    n = len(x)
    if n == 0:
        return x
    starts = np.flatnonzero(np.diff(np.r_[0, x]) == 1)
    stops = np.flatnonzero(np.diff(np.r_[x, 0]) == -1)
    for s, t in zip(starts, stops):
        if (t - s) < min_len:
            x[s:t] = 0
    return x


def remove_short_events(bin_df: pd.DataFrame, video_id: pd.Series, min_len: int) -> pd.DataFrame:
    if min_len <= 1:
        return bin_df
    out_parts = []
    for vid, g in bin_df.assign(video_id=video_id.values).groupby("video_id"):
        arr = g.drop(columns="video_id").to_numpy(copy=True)
        for j in range(arr.shape[1]):
            arr[:, j] = _remove_short_ones(arr[:, j], min_len)
        out = pd.DataFrame(arr, index=g.index, columns=bin_df.columns)
        out_parts.append(out)
    return pd.concat(out_parts, axis=0).sort_index()


def combine_factions(df: pd.DataFrame, w: dict) -> pd.Series:
    num = None
    denom = 0.0
    for k, v in w.items():
        if k in df.columns and v > 0:
            num = df[k] * v if num is None else num + df[k] * v
            denom += v
    if num is None:
        raise ValueError("No valid faction columns in df")
    return num / max(denom, 1e-9)


def load_oof_files(oof_dir: str, switch: str, action: str) -> pd.DataFrame:
    paths = sorted(glob(f"{oof_dir}/oof_*_{switch}_{action}_fold*.pkl"))
    if len(paths) == 0:
        raise FileNotFoundError(f"OOF not found for switch={switch}, action={action}")
    dfs = [pd.DataFrame(joblib.load(p)) for p in paths]
    return pd.concat(dfs, axis=0, ignore_index=True)


def make_objective(oof_df: pd.DataFrame, action: str,
                   smooth_choices=range(1, 62, 2),
                   min_event_choices=range(1, 8)):
    def objective(trial: optuna.Trial) -> float:
        # Dirichlet 重み（不足列は 0 扱い）
        w_raw = trial.dirichlet("w", [1, 1, 1, 1])
        factions = ["pred_lgbm", "pred_xgb", "pred_cat", "pred_rf"]
        w = {k: w_raw[i] for i, k in enumerate(factions)}

        thr = trial.suggest_float("threshold", 0.05, 0.95)
        smooth_w = trial.suggest_categorical("smooth_w", list(smooth_choices))
        min_len = trial.suggest_categorical("min_event_len", list(min_event_choices))

        prob_cols = [c for c in factions if c in oof_df.columns]
        proba = combine_factions(oof_df[prob_cols], w)

        # 平滑化（video_id ごと）
        smoothed = smooth_by_video(proba.to_frame(action), oof_df["video_id"], smooth_w)[action]
        bin_pred = (smoothed >= thr).astype(int)
        bin_pred = remove_short_events(bin_pred.to_frame(action), oof_df["video_id"], min_len)[action]

        score = f1_score(oof_df["y_true"], bin_pred, average="macro", zero_division=0)
        return score
    return objective


def run_optuna_for_action(oof_dir: str, switch: str, action: str,
                          n_trials: int = 50, storage: str | None = None,
                          study_name: str | None = None):
    oof_df = load_oof_files(oof_dir, switch, action)
    obj = make_objective(oof_df, action)
    study = optuna.create_study(direction="maximize", storage=storage, study_name=study_name, load_if_exists=bool(storage))
    study.optimize(obj, n_trials=n_trials, show_progress_bar=False)
    print(f"best_value={study.best_value:.5f}")
    print(f"best_params={study.best_params}")
    return study


def load_pred_files(pred_dir: str, switch: str, action: str) -> pd.DataFrame:
    paths = sorted(glob(f"{pred_dir}/pred_*_{switch}_{action}_fold*.pkl"))
    if len(paths) == 0:
        raise FileNotFoundError(f"pred not found for switch={switch}, action={action}")
    dfs = [pd.DataFrame(joblib.load(p)) for p in paths]
    return pd.concat(dfs, axis=0, ignore_index=True)


def blend_pred_with_weights(pred_df: pd.DataFrame, w: dict) -> pd.Series:
    factions = ["pred_lgbm", "pred_xgb", "pred_cat", "pred_rf"]
    prob_cols = [c for c in factions if c in pred_df.columns]
    if len(prob_cols) == 0:
        raise ValueError("no pred_* columns in pred_df")
    return combine_factions(pred_df[prob_cols], {k: w.get(k, 0.0) for k in factions})



In [ ]:
# ==================== Optuna 実行ブロック（OOF 完了後に手動実行） ====================
RUN_OPTUNA = os.environ.get("RUN_OPTUNA", "False").lower() == "true"
OPTUNA_TRIALS = int(os.environ.get("OPTUNA_TRIALS", 50))
OOF_DIR = os.environ.get("SAVE_OOF_DIR", "oof")
OPTUNA_BEST_PATH = os.environ.get("OPTUNA_BEST_PATH", "optuna_best.json")

if RUN_OPTUNA:
    print("Starting Optuna optimization...")
    best_params_dict = {}

    for switch in ["single", "pair"]:
        pattern = f"{OOF_DIR}/oof_*_{switch}_*_fold*.pkl"
        files = glob(pattern)
        actions = sorted({Path(p).stem.split(f"_{switch}_", 1)[1].rsplit("_fold", 1)[0] for p in files})
        for action in actions:
            try:
                study = run_optuna_for_action(OOF_DIR, switch, action, n_trials=OPTUNA_TRIALS)
                best_params_dict[action] = study.best_params
            except Exception as e:
                print(f"[Optuna] skip {switch}-{action} due to {e}")

    with open(OPTUNA_BEST_PATH, "w") as f:
        json.dump(best_params_dict, f, indent=2)
    print(f"Saved optuna_best to {OPTUNA_BEST_PATH}")



In [ ]:
def robustify(submission, dataset, traintest, traintest_directory=None):
    if traintest_directory is None:
        traintest_directory = f"/kaggle/input/MABe-mouse-behavior-detection/{traintest}_tracking"

    submission = submission[submission.start_frame < submission.stop_frame]

    group_list = []
    for _, group in submission.groupby(['video_id', 'agent_id', 'target_id']):
        group = group.sort_values('start_frame')
        mask = np.ones(len(group), dtype=bool)
        last_stop = 0
        for i, (_, row) in enumerate(group.iterrows()):
            if row['start_frame'] < last_stop:
                mask[i] = False
            else:
                last_stop = row['stop_frame']
        group_list.append(group[mask])
    submission = pd.concat(group_list) if group_list else submission

    s_list = []
    for _, row in dataset.iterrows():
        lab_id = row['lab_id']
        video_id = row['video_id']
        if (submission.video_id == video_id).any():
            continue

        if verbose:
            print(f"Video {video_id} has no predictions")

        path = f"{traintest_directory}/{lab_id}/{video_id}.parquet"
        vid = pd.read_parquet(path)

        vid_behaviors = eval(row['behaviors_labeled'])
        vid_behaviors = sorted(list({b.replace("'", "") for b in vid_behaviors}))
        vid_behaviors = [b.split(',') for b in vid_behaviors]
        vid_behaviors = pd.DataFrame(vid_behaviors, columns=['agent', 'target', 'action'])

        start_frame = vid.video_frame.min()
        stop_frame = vid.video_frame.max() + 1

        for (agent, target), actions in vid_behaviors.groupby(['agent', 'target']):
            batch_len = int(np.ceil((stop_frame - start_frame) / len(actions)))
            for i, (_, action_row) in enumerate(actions.iterrows()):
                batch_start = start_frame + i * batch_len
                batch_stop = min(batch_start + batch_len, stop_frame)
                s_list.append((video_id, agent, target, action_row['action'], batch_start, batch_stop))

    if len(s_list) > 0:
        submission = pd.concat([
            submission,
            pd.DataFrame(s_list, columns=['video_id', 'agent_id', 'target_id', 'action', 'start_frame', 'stop_frame'])
        ])

    submission = submission.reset_index(drop=True)
    return submission

In [ ]:
# ==================== MAIN LOOP ====================
if SKIP_SMALL_TEST:
    # 最低限 1 行は入れたサブミッションを出力して終了（学習・推論は実行しない）
    if len(test) > 0:
        vid0 = test['video_id'].iloc[0]
    else:
        vid0 = 438887472
    submission = pd.DataFrame({
        'video_id': [vid0],
        'agent_id': ['mouse1'],
        'target_id': ['self'],
        'action': ['rear'],
        'start_frame': [278],
        'stop_frame': [500],
    })
    submission.index.name = 'row_id'
    submission.to_csv('submission.csv')
    print(f"test.csv が {len(test)} 行のため、学習/推論をスキップして submission.csv を出力しました (行数: {len(submission)}).")
else:
    MODE = os.environ.get("MODE", "submit")
    SAVE_OOF_DIR = os.environ.get("SAVE_OOF_DIR", "oof")
    SAVE_PRED_DIR = os.environ.get("SAVE_PRED_DIR", "pred")

    submission_list = []

    for section in range(len(body_parts_tracked_list)):
        body_parts_tracked_str = body_parts_tracked_list[section]
        try:
            body_parts_tracked = json.loads(body_parts_tracked_str)
            print(f"{section}. Processing: {len(body_parts_tracked)} body parts")
            if len(body_parts_tracked) > 5:
                body_parts_tracked = [b for b in body_parts_tracked if b not in drop_body_parts]

            train_subset = train[train.body_parts_tracked == body_parts_tracked_str]

            _fps_lookup = (
                train_subset[['video_id', 'frames_per_second']]
                .drop_duplicates('video_id')
                .set_index('video_id')['frames_per_second']
                .to_dict()
            )

            single_list, single_label_list, single_meta_list = [], [], []
            pair_list, pair_label_list, pair_meta_list = [], [], []

            for switch, data, meta, label in generate_mouse_data(train_subset, 'train'):
                if switch == 'single':
                    single_list.append(data)
                    single_meta_list.append(meta)
                    single_label_list.append(label)
                else:
                    pair_list.append(data)
                    pair_meta_list.append(meta)
                    pair_label_list.append(label)

            if len(single_list) > 0:
                single_feats_parts = []
                for data_i, meta_i in zip(single_list, single_meta_list):
                    fps_i = _fps_from_meta(meta_i, _fps_lookup, default_fps=30.0)
                    Xi = transform_single(data_i, body_parts_tracked, fps_i, meta_df=meta_i).astype(np.float32)
                    single_feats_parts.append(Xi)

                X_tr = pd.concat(single_feats_parts, axis=0, ignore_index=True)
    
                single_label = pd.concat(single_label_list, axis=0, ignore_index=True)
                single_meta  = pd.concat(single_meta_list,  axis=0, ignore_index=True)

                del single_list, single_label_list, single_meta_list, single_feats_parts
                gc.collect()

                print(f"  Single: {X_tr.shape}")
                if MODE == "submit":
                    submit_ensemble(body_parts_tracked_str, 'single', X_tr, single_label, single_meta)
                else:
                    submit_ensemble_cv(
                        body_parts_tracked_str,
                        'single',
                        X_tr,
                        single_label,
                        single_meta,
                        n_splits=N_SPLITS,
                        my_fold=MY_FOLD,
                        save_dir=SAVE_OOF_DIR,
                        pred_dir=SAVE_PRED_DIR,
                    )

                del X_tr, single_label, single_meta
                gc.collect()

            if len(pair_list) > 0:
                pair_feats_parts = []
                for data_i, meta_i in zip(pair_list, pair_meta_list):
                    fps_i = _fps_from_meta(meta_i, _fps_lookup, default_fps=30.0)
                    Xi = transform_pair(data_i, body_parts_tracked, fps_i, meta_df=meta_i).astype(np.float32)
                    pair_feats_parts.append(Xi)

                X_tr = pd.concat(pair_feats_parts, axis=0, ignore_index=True)

                
                pair_label = pd.concat(pair_label_list, axis=0, ignore_index=True)
                pair_meta  = pd.concat(pair_meta_list,  axis=0, ignore_index=True)

                del pair_list, pair_label_list, pair_meta_list, pair_feats_parts
                gc.collect()

                print(f"  Pair: {X_tr.shape}")
                if MODE == "submit":
                    submit_ensemble(body_parts_tracked_str, 'pair', X_tr, pair_label, pair_meta)
                else:
                    submit_ensemble_cv(
                        body_parts_tracked_str,
                        'pair',
                        X_tr,
                        pair_label,
                        pair_meta,
                        n_splits=N_SPLITS,
                        my_fold=MY_FOLD,
                        save_dir=SAVE_OOF_DIR,
                        pred_dir=SAVE_PRED_DIR,
                    )

                del X_tr, pair_label, pair_meta
                gc.collect()

        except Exception as e:
            print(f'***Exception*** {str(e)[:100]}')

        gc.collect()
        print()

    if MODE == "submit":
        if len(submission_list) > 0:
            submission = pd.concat(submission_list, ignore_index=True)
        else:
            submission = pd.DataFrame({
                'video_id': [438887472],
                'agent_id': ['mouse1'],
                'target_id': ['self'],
                'action': ['rear'],
                'start_frame': [278],
                'stop_frame': [500]
            })

        submission_robust = robustify(submission, test, 'test')
        submission_robust.index.name = 'row_id'
        submission_robust.to_csv('submission.csv')
        print(f"\nSubmission created: {len(submission_robust)} predictions")
    else:
        print("MODE!=submit のため submission.csv は作成しません (OOF のみ出力)。")
